# Gaussian Beam in Water — GPU Simulation

Same physical scenario as `01_gaussian_beam_cpu.ipynb` but using the GPU backends
(`cupy` vectorized arrays or `numba` CUDA JIT kernel). Includes a timing comparison
against the CPU baseline.

**Requirements**

- NVIDIA GPU with CUDA 12+
- `cupy` and/or `numba` installed (uncomment in `environment.yml` and run
  `micromamba env update -f environment.yml`)

If no GPU is available the notebook falls back to the CPU backend automatically
and the timing comparison uses CPU-only numbers.

**Physical scenario** — identical to notebook 01:
clear ocean water at 532 nm, 16 m range, 0.8 m aperture, 90° FOV.

## 1. Imports and GPU detection

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from photonator import Simulation
from photonator.beam.gaussian import GaussianBeam
from photonator.media.water import Water
from photonator.phase_functions.petzold import PetzoldPhaseFunction
from photonator.core.receiver import Receiver
from photonator.io.hdf5 import save_hdf5

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# ── Detect available GPU backends ─────────────────────────────────────────────
def detect_gpu():
    """Return the best available GPU backend, or 'cpu' if none found."""
    try:
        import cupy as cp
        dev = cp.cuda.Device(0)
        name = dev.pci_bus_id
        mem_gb = dev.mem_info[1] / 1024**3
        print(f"CuPy found: device {name}, {mem_gb:.1f} GB VRAM")
        return "cupy"
    except Exception:
        pass
    try:
        from numba import cuda
        if cuda.is_available():
            gpu = cuda.get_current_device()
            print(f"Numba CUDA found: {gpu.name.decode()}")
            return "numba"
    except Exception:
        pass
    print("No GPU found — falling back to CPU backend.")
    return "cpu"

GPU_BACKEND = detect_gpu()
print(f"Using backend: {GPU_BACKEND!r}")

## 2. Build simulation components

Identical setup to notebook 01 — components are backend-agnostic.

In [ ]:
water = Water(
    mu_a_per_m=0.0088,
    mu_s_per_m=0.037,
    g=0.93,
    turbidity=1.0,
    wavelength_nm=532.0,
)

beam = GaussianBeam(
    w0_m=0.001,
    half_angle_divergence_rad=0.00075,
)

phase_fn = PetzoldPhaseFunction(water_type="clear")

RANGE_M = 16.0
receiver = Receiver(
    receiver_z_m=RANGE_M,
    aperture_m=0.8,
    fov_rad=np.pi / 2,
)

c = water.mu_a_per_m + water.mu_s_per_m
print(f"c = {c:.4f} m⁻¹, range = {RANGE_M} m ({RANGE_M * c:.2f} att. lengths)")

## 3. GPU simulation — 1 M photons

In [ ]:
N_PHOTONS_GPU = 1_000_000   # 1 M photons per batch
N_BATCHES_GPU = 5           # 5 M total

sim_gpu = Simulation(
    medium=water,
    beam=beam,
    phase_fn=phase_fn,
    receiver=receiver,
    n_photons=N_PHOTONS_GPU,
    n_batches=N_BATCHES_GPU,
    backend=GPU_BACKEND,
    store_photons=True,
    seed=42,
)

print(f"Running {N_PHOTONS_GPU * N_BATCHES_GPU:,} photons on backend={GPU_BACKEND!r} …")
t0 = time.perf_counter()
result_gpu = sim_gpu.run()
gpu_time = time.perf_counter() - t0
print(f"Done in {gpu_time:.2f} s  "
      f"({N_PHOTONS_GPU * N_BATCHES_GPU / gpu_time / 1e6:.2f} M photons/s)")

## 4. CPU baseline — same photon count

In [ ]:
N_PHOTONS_CPU = 100_000    # 100 k per batch (GPU run was 1 M, but CPU scales linearly)
N_BATCHES_CPU = 5          # 500 k total

receiver_cpu = Receiver(receiver_z_m=RANGE_M, aperture_m=0.8, fov_rad=np.pi / 2)

sim_cpu = Simulation(
    medium=water,
    beam=beam,
    phase_fn=phase_fn,
    receiver=receiver_cpu,
    n_photons=N_PHOTONS_CPU,
    n_batches=N_BATCHES_CPU,
    backend="cpu",
    seed=42,
)

print(f"Running {N_PHOTONS_CPU * N_BATCHES_CPU:,} photons on backend='cpu' …")
t0 = time.perf_counter()
result_cpu = sim_cpu.run()
cpu_time = time.perf_counter() - t0
cpu_rate = N_PHOTONS_CPU * N_BATCHES_CPU / cpu_time
print(f"Done in {cpu_time:.2f} s  ({cpu_rate / 1e6:.2f} M photons/s)")

# Extrapolate CPU time to the same photon count as GPU run
cpu_time_equiv = (N_PHOTONS_GPU * N_BATCHES_GPU) / cpu_rate
speedup = cpu_time_equiv / gpu_time if gpu_time > 0 else float("inf")
print(f"\nEquivalent CPU time for {N_PHOTONS_GPU * N_BATCHES_GPU / 1e6:.0f} M photons: "
      f"{cpu_time_equiv:.1f} s")
print(f"GPU speedup estimate: {speedup:.1f}×")

## 5. Timing comparison chart

In [ ]:
backends = ["CPU (NumPy)", f"GPU ({GPU_BACKEND})"]
rates = [cpu_rate / 1e6,
         (N_PHOTONS_GPU * N_BATCHES_GPU) / gpu_time / 1e6]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(backends, rates, color=["steelblue", "tomato"], width=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{rate:.2f} M/s", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Throughput  (M photons / s)")
ax.set_title("CPU vs GPU throughput — Gaussian beam in water")
ax.set_ylim(0, max(rates) * 1.25)
plt.tight_layout()
plt.show()

## 6. Results comparison — CPU vs GPU

Both backends must produce statistically consistent results for the same scenario.
Differences should be within shot-noise (1/√N).

In [ ]:
beer_lambert = np.exp(-c * RANGE_M)

cpu_frac = result_cpu.total_power / result_cpu.n_photons
gpu_frac = result_gpu.total_power / result_gpu.n_photons

print(f"{'Metric':<30} {'CPU':>14} {'GPU':>14}")
print("-" * 60)
print(f"{'Photons launched':<30} {result_cpu.n_photons:>14,} {result_gpu.n_photons:>14,}")
print(f"{'Photons detected':<30} {result_cpu.total_packets:>14,} {result_gpu.total_packets:>14,}")
print(f"{'Detected fraction':<30} {cpu_frac:>14.6f} {gpu_frac:>14.6f}")
print(f"{'Beer-Lambert (ballistic)':<30} {beer_lambert:>14.6f} {beer_lambert:>14.6f}")
print(f"{'Mean arrival uz':<30} {result_cpu.angle_mean_rad:>14.6f} {result_gpu.angle_mean_rad:>14.6f}")
print(f"{'Mean path length (m)':<30} {result_cpu.dist_mean_m:>14.4f} {result_gpu.dist_mean_m:>14.4f}")
print(f"{'Elapsed time (s)':<30} {result_cpu.elapsed_s:>14.2f} {result_gpu.elapsed_s:>14.2f}")

## 7. Arrival position comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_range = [[-0.4, 0.4], [-0.4, 0.4]]
bins = 80

for ax, res, label, cmap in zip(
    axes,
    [result_cpu, result_gpu],
    ["CPU (NumPy)", f"GPU ({GPU_BACKEND})"],
    ["Blues", "Reds"],
):
    if res.rec_loc is not None:
        x = res.rec_loc[:, 0]
        y = res.rec_loc[:, 1]
        h = ax.hist2d(x, y, bins=bins, range=plot_range,
                      norm=mcolors.LogNorm(), cmap=cmap)
        plt.colorbar(h[3], ax=ax, label="Photon count")
    ax.set_xlabel("x  (m)")
    ax.set_ylabel("y  (m)")
    ax.set_title(f"Arrival positions — {label}\n"
                 f"({res.n_photons:,} photons launched)")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 8. Path length distributions — CPU vs GPU

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for res, label, color in [
    (result_cpu, "CPU", "steelblue"),
    (result_gpu, f"GPU ({GPU_BACKEND})", "tomato"),
]:
    if res.distances_m is not None:
        ax.hist(res.distances_m, bins=120, density=True,
                alpha=0.6, color=color, label=label, edgecolor="none")

ax.axvline(RANGE_M, color="black", lw=1.5, linestyle="--",
           label=f"geometric path = {RANGE_M} m")
ax.set_xlabel("Path length  (m)")
ax.set_ylabel("Probability density")
ax.set_title("Path length distribution — CPU vs GPU")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Save GPU results to HDF5

In [ ]:
output_path = "gaussian_beam_gpu.h5"
save_hdf5(result_gpu, output_path)
print(f"Saved GPU result to {output_path}")